In [6]:
from dotenv import load_dotenv
import os
import time
import cfbd
import pandas as pd
from pathlib import Path
import requests

In [19]:
load_dotenv()

API_KEY = os.environ.get("CFBD_API_KEY")

if API_KEY is None:
    raise ValueError("CFBD_API_KEY not set in environment variables")

SEASONS = [2024]
OUTPUT_DIR = Path("test_data")
OUTPUT_DIR.mkdir(exist_ok=True)
SLEEP = 1.0

In [15]:
# API Setup
configuration = cfbd.Configuration()
configuration.access_token = API_KEY
# configuration.api_key['Authorization'] = API_KEY
configuration.api_key_prefix['Authorization'] = "Bearer"

def get_client():
    # print(cfbd.ApiClient(configuration))
    return cfbd.ApiClient(configuration)

def check_api_quota():
    with get_client() as client:
        info_api = cfbd.InfoApi(client)
        info = info_api.get_user_info()
        print(f" API quota remaining: {info.usage.calls_remaining} calls")

## Data Collection

In [40]:
def fetch_sp_lines(seasons: list[int], api_key: str) -> pd.DataFrame:
    """
    Pull SP+ pregame spread per game per 2024 season.
    """
    print("\nFetching SP+ pregame win probabilities...")
    records = []
    headers = {"Authorization": f"Bearer {api_key}"}

    for season in seasons:
        print(f"  Season {season}...", end=" ")
        try:
            response = requests.get(
                "https://api.collegefootballdata.com/metrics/wp/pregame",
                headers=headers,
                params={"year": season}
            )
            response.raise_for_status()
            data = response.json()

            for entry in data:
                records.append({
                    "game_id":        entry.get("gameId"),
                    "season":         season,
                    "home_team":      entry.get("homeTeam"),
                    "away_team":      entry.get("awayTeam"),
                    "sp_spread":         entry.get("spread"),
                })
            print(f"{len(data)} games")

        except requests.HTTPError as e:
            print(f"HTTP ERROR: {e}")
        time.sleep(SLEEP)

    df = pd.DataFrame(records)
    print(f"  Total: {len(df):,} game win probability records")
    return df


In [ ]:
def fetch_betting_lines(seasons: list[int], api_key: str) -> pd.DataFrame:
    """
    Pull Vegas betting lines per game.
    """
    print("\nFetching betting lines...")
    records = []
    headers = {"Authorization": f"Bearer {api_key}"}

    for season in seasons:
        print(f"  Season {season}...", end=" ")
        try:
            response = requests.get(
                "https://api.collegefootballdata.com/lines",
                headers=headers,
                params={
                    "year":        season,
                    "seasonType":  "regular"
                }
            )
            response.raise_for_status()
            data = response.json()

            for game in data:
                # Multiple providers per game — filter to Consensus or average
                lines = game.get("lines", [])
                
                # Try to get consensus line first, fall back to first available
                consensus = next(
                    (l for l in lines if l.get("provider", "").lower() == "DraftKings"),
                    lines[0] if lines else None
                )

                if consensus:
                    spread = consensus.get("spread")

                    records.append({
                        "game_id":              game.get("id"),
                        "season":               season,
                        "home_team":            game.get("homeTeam"),
                        "away_team":            game.get("awayTeam"),
                        "vegas_spread":         spread,
                        "provider":             consensus.get("provider"),
                    })

        except requests.HTTPError as e:
            print(f"HTTP ERROR: {e}")
        time.sleep(SLEEP)
    print(f" Total games: {len(records)}")
    df = pd.DataFrame(records)
    return df

In [45]:
# get the vegas spreads
df_vegas = fetch_betting_lines(seasons=SEASONS, api_key=API_KEY)
df_vegas = df_vegas[df_vegas['provider'].str.lower() == "draftkings"]

# get the sp+ spreads
df_sp = fetch_sp_lines(seasons=SEASONS, api_key=API_KEY)

# merge to pull the spreads together
df_merge = df_sp.merge(df_vegas, on=['game_id', 'season', 'home_team', 'away_team'], how="inner")

# pull in the actuals
df_actuals = pd.read_csv("data/games.csv")
df_actuals = df_actuals[["id", "season", "homeTeam", "awayTeam", "homePoints", "awayPoints"]].rename(columns={
    "id": "game_id", 
    "homeTeam": "home_team", 
    "awayTeam": "away_team"
})
df_actuals['actual_spread'] = df_actuals['homePoints'] - df_actuals['awayPoints']
df_actuals = df_actuals.drop(columns=["homePoints", "awayPoints"])

# merge in the actuals
df_final = df_merge.merge(df_actuals, on=["game_id", "season", "home_team", "away_team"], how="inner")
df_final["sp_spread"] = df_final["sp_spread"] * -1
df_final["vegas_spread"] = df_final["vegas_spread"] * -1

# load to csv
df_final.to_csv(OUTPUT_DIR / "2024_spreads.csv", index=False)


Fetching betting lines...
  Season 2024... 1523 games
  Total: 1,507 betting line records

Fetching SP+ pregame win probabilities...
  Season 2024... 825 games
  Total: 825 game win probability records


/var/folders/y0/6_l3s2r147zb5njrsxhlx0b40000gn/T/ipykernel_37808/2879955051.py:12: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df_actuals = pd.read_csv("data/games.csv")
